In [17]:
import pandas as pd
import tensorflow as tf

SHUFFLE_BUFFER = 500
BATCH_SIZE = 2

In [18]:
#YOU HAVE TO HAVE THE CSV IN SAME DIRECTORY 

csv_file_path = 'forestfires.csv' 

In [19]:
df = pd.read_csv(csv_file_path)

In [20]:
df.head()

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0


In [21]:
df.dtypes

X          int64
Y          int64
month     object
day       object
FFMC     float64
DMC      float64
DC       float64
ISI      float64
temp     float64
RH         int64
wind     float64
rain     float64
area     float64
dtype: object

In [22]:
target = df.pop('area')

In [24]:
numeric_feature_names = ['rain', 'wind',  'RH', 'temp']
numeric_features = df[numeric_feature_names]
numeric_features.head()

,rain,wind,RH,temp
0,0.0,6.7,51,8.2
1,0.0,0.9,33,18.0
2,0.0,1.3,33,14.6
3,0.2,4.0,97,8.3
4,0.0,1.8,99,11.4


In [25]:
tf.convert_to_tensor(numeric_features)

<tf.Tensor: shape=(517, 4), dtype=float64, numpy=
array([[ 0. ,  6.7, 51. ,  8.2],
       [ 0. ,  0.9, 33. , 18. ],
       [ 0. ,  1.3, 33. , 14.6],
       ...,
       [ 0. ,  6.7, 70. , 21.2],
       [ 0. ,  4. , 42. , 25.6],
       [ 0. ,  4.5, 31. , 11.8]])>

In [26]:
normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(numeric_features)

In [27]:
normalizer(numeric_features.iloc[:3])

<tf.Tensor: shape=(3, 4), dtype=float32, numpy=
array([[-0.07326832,  1.498614  ,  0.41172418, -1.8426397 ],
       [-0.07326832, -1.7417557 , -0.6924565 , -0.15327784],
       [-0.07326832, -1.518282  , -0.6924565 , -0.73938286]],
      dtype=float32)>

In [28]:
def get_basic_model():
  model = tf.keras.Sequential([
    normalizer,
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(1)
  ])

  model.compile(optimizer='adam',
                loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
                metrics=['accuracy'])
  return model

In [29]:
model = get_basic_model()
model.fit(numeric_features, target, epochs=15, batch_size=BATCH_SIZE)

Epoch 1/15
259/259 [==============================] - 1s 842us/step - loss: -6.9038 - accuracy: 0.2611
Epoch 2/15
259/259 [==============================] - 0s 846us/step - loss: -24.9328 - accuracy: 0.0000e+00
Epoch 3/15
259/259 [==============================] - 0s 838us/step - loss: -51.0620 - accuracy: 0.0000e+00
Epoch 4/15
259/259 [==============================] - 0s 816us/step - loss: -100.8563 - accuracy: 0.0000e+00
Epoch 5/15
259/259 [==============================] - 0s 831us/step - loss: -176.8490 - accuracy: 0.0000e+00
Epoch 6/15
259/259 [==============================] - 0s 832us/step - loss: -276.6789 - accuracy: 0.0000e+00
Epoch 7/15
259/259 [==============================] - 0s 834us/step - loss: -410.3959 - accuracy: 0.0000e+00
Epoch 8/15
259/259 [==============================] - 0s 823us/step - loss: -576.7502 - accuracy: 0.0000e+00
Epoch 9/15
259/259 [==============================] - 0s 823us/step - loss: -789.9270 - accuracy: 0.0000e+00
Epoch 10/15
259/259 [======

In [30]:
numeric_dataset = tf.data.Dataset.from_tensor_slices((numeric_features, target))

for row in numeric_dataset.take(3):
  print(row)

(<tf.Tensor: shape=(4,), dtype=float64, numpy=array([ 0. ,  6.7, 51. ,  8.2])>, <tf.Tensor: shape=(), dtype=float64, numpy=0.0>)
(<tf.Tensor: shape=(4,), dtype=float64, numpy=array([ 0. ,  0.9, 33. , 18. ])>, <tf.Tensor: shape=(), dtype=float64, numpy=0.0>)
(<tf.Tensor: shape=(4,), dtype=float64, numpy=array([ 0. ,  1.3, 33. , 14.6])>, <tf.Tensor: shape=(), dtype=float64, numpy=0.0>)


In [31]:
numeric_batches = numeric_dataset.shuffle(1000).batch(BATCH_SIZE)

model = get_basic_model()
model.fit(numeric_batches, epochs=15)

Epoch 1/15
259/259 [==============================] - 1s 817us/step - loss: -4.0681 - accuracy: 0.2921
Epoch 2/15
259/259 [==============================] - 0s 844us/step - loss: -14.3411 - accuracy: 0.0213
Epoch 3/15
259/259 [==============================] - 0s 826us/step - loss: -46.4067 - accuracy: 0.0000e+00
Epoch 4/15
259/259 [==============================] - 0s 833us/step - loss: -106.2121 - accuracy: 0.0000e+00
Epoch 5/15
259/259 [==============================] - 0s 855us/step - loss: -208.5445 - accuracy: 0.0000e+00
Epoch 6/15
259/259 [==============================] - 0s 862us/step - loss: -371.4477 - accuracy: 0.0000e+00
Epoch 7/15
259/259 [==============================] - 0s 869us/step - loss: -597.3591 - accuracy: 0.0000e+00
Epoch 8/15
259/259 [==============================] - 0s 892us/step - loss: -867.0883 - accuracy: 0.0000e+00
Epoch 9/15
259/259 [==============================] - 0s 874us/step - loss: -1201.8636 - accuracy: 0.0000e+00
Epoch 10/15
259/259 [=========

In [32]:
numeric_dict_ds = tf.data.Dataset.from_tensor_slices((dict(numeric_features), target))

In [33]:
for row in numeric_dict_ds.take(3):
  print(row)

({'rain': <tf.Tensor: shape=(), dtype=float64, numpy=0.0>, 'wind': <tf.Tensor: shape=(), dtype=float64, numpy=6.7>, 'RH': <tf.Tensor: shape=(), dtype=int64, numpy=51>, 'temp': <tf.Tensor: shape=(), dtype=float64, numpy=8.2>}, <tf.Tensor: shape=(), dtype=float64, numpy=0.0>)
({'rain': <tf.Tensor: shape=(), dtype=float64, numpy=0.0>, 'wind': <tf.Tensor: shape=(), dtype=float64, numpy=0.9>, 'RH': <tf.Tensor: shape=(), dtype=int64, numpy=33>, 'temp': <tf.Tensor: shape=(), dtype=float64, numpy=18.0>}, <tf.Tensor: shape=(), dtype=float64, numpy=0.0>)
({'rain': <tf.Tensor: shape=(), dtype=float64, numpy=0.0>, 'wind': <tf.Tensor: shape=(), dtype=float64, numpy=1.3>, 'RH': <tf.Tensor: shape=(), dtype=int64, numpy=33>, 'temp': <tf.Tensor: shape=(), dtype=float64, numpy=14.6>}, <tf.Tensor: shape=(), dtype=float64, numpy=0.0>)


In [34]:
  def stack_dict(inputs, fun=tf.stack):
    values = []
    for key in sorted(inputs.keys()):
      values.append(tf.cast(inputs[key], tf.float32))

    return fun(values, axis=-1)

In [35]:
#@title
class MyModel(tf.keras.Model):
  def __init__(self):
    # Create all the internal layers in init.
    super().__init__()

    self.normalizer = tf.keras.layers.Normalization(axis=-1)

    self.seq = tf.keras.Sequential([
      self.normalizer,
      tf.keras.layers.Dense(10, activation='relu'),
      tf.keras.layers.Dense(10, activation='relu'),
      tf.keras.layers.Dense(1)
    ])

  def adapt(self, inputs):
    # Stack the inputs and `adapt` the normalization layer.
    inputs = stack_dict(inputs)
    self.normalizer.adapt(inputs)

  def call(self, inputs):
    # Stack the inputs
    inputs = stack_dict(inputs)
    # Run them through all the layers.
    result = self.seq(inputs)

    return result

model = MyModel()

model.adapt(dict(numeric_features))

model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'],
              run_eagerly=True)

In [36]:
model.fit(dict(numeric_features), target, epochs=5, batch_size=BATCH_SIZE)

Epoch 1/5
259/259 [==============================] - 6s 21ms/step - loss: -2.0835 - accuracy: 0.3598
Epoch 2/5
259/259 [==============================] - 5s 21ms/step - loss: -13.8149 - accuracy: 0.0600
Epoch 3/5
259/259 [==============================] - 5s 21ms/step - loss: -35.9519 - accuracy: 0.0000e+00
Epoch 4/5
259/259 [==============================] - 5s 21ms/step - loss: -71.6913 - accuracy: 0.0000e+00
Epoch 5/5
259/259 [==============================] - 6s 21ms/step - loss: -122.7574 - accuracy: 0.0000e+00


In [37]:
numeric_dict_batches = numeric_dict_ds.shuffle(SHUFFLE_BUFFER).batch(BATCH_SIZE)
model.fit(numeric_dict_batches, epochs=5)

Epoch 1/5
259/259 [==============================] - 5s 19ms/step - loss: -208.3504 - accuracy: 0.0000e+00
Epoch 2/5
259/259 [==============================] - 5s 20ms/step - loss: -330.6938 - accuracy: 0.0000e+00
Epoch 3/5
259/259 [==============================] - 5s 19ms/step - loss: -477.0215 - accuracy: 0.0000e+00
Epoch 4/5
259/259 [==============================] - 5s 19ms/step - loss: -661.4548 - accuracy: 0.0000e+00
Epoch 5/5
259/259 [==============================] - 5s 19ms/step - loss: -874.1346 - accuracy: 0.0000e+00


In [38]:
model.predict(dict(numeric_features.iloc[:3]))

1/1 [==============================] - 0s 24ms/step


array([[[45.71879 ]],

       [[64.19921 ]],

       [[50.614925]]], dtype=float32)

In [39]:
inputs = {}
for name, column in numeric_features.items():
  inputs[name] = tf.keras.Input(
      shape=(1,), name=name, dtype=tf.float32)

inputs

{'rain': <KerasTensor: shape=(None, 1) dtype=float32 (created by layer 'rain')>,
 'wind': <KerasTensor: shape=(None, 1) dtype=float32 (created by layer 'wind')>,
 'RH': <KerasTensor: shape=(None, 1) dtype=float32 (created by layer 'RH')>,
 'temp': <KerasTensor: shape=(None, 1) dtype=float32 (created by layer 'temp')>}

In [40]:
x = stack_dict(inputs, fun=tf.concat)

normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(stack_dict(dict(numeric_features)))

x = normalizer(x)
x = tf.keras.layers.Dense(10, activation='relu')(x)
x = tf.keras.layers.Dense(10, activation='relu')(x)
x = tf.keras.layers.Dense(1)(x)

model = tf.keras.Model(inputs, x)

model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'],
              run_eagerly=True)

In [41]:
tf.keras.utils.plot_model(model, rankdir="LR", show_shapes=True)

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [42]:
model.fit(dict(numeric_features), target, epochs=5, batch_size=BATCH_SIZE)

Epoch 1/5
259/259 [==============================] - 5s 19ms/step - loss: -3.9450 - accuracy: 0.3308
Epoch 2/5
259/259 [==============================] - 5s 19ms/step - loss: -16.0300 - accuracy: 0.0851
Epoch 3/5
259/259 [==============================] - 5s 19ms/step - loss: -40.7359 - accuracy: 0.0000e+00
Epoch 4/5
259/259 [==============================] - 5s 20ms/step - loss: -81.0440 - accuracy: 0.0000e+00
Epoch 5/5
259/259 [==============================] - 5s 20ms/step - loss: -142.7004 - accuracy: 0.0000e+00


In [43]:
numeric_dict_batches = numeric_dict_ds.shuffle(SHUFFLE_BUFFER).batch(BATCH_SIZE)
model.fit(numeric_dict_batches, epochs=5)

Epoch 1/5
259/259 [==============================] - 5s 20ms/step - loss: -233.1963 - accuracy: 0.0000e+00
Epoch 2/5
259/259 [==============================] - 5s 20ms/step - loss: -352.8259 - accuracy: 0.0000e+00
Epoch 3/5
259/259 [==============================] - 5s 19ms/step - loss: -523.4308 - accuracy: 0.0000e+00
Epoch 4/5
259/259 [==============================] - 5s 20ms/step - loss: -734.4836 - accuracy: 0.0000e+00
Epoch 5/5
259/259 [==============================] - 5s 20ms/step - loss: -988.0673 - accuracy: 0.0000e+00


#FULL EXAMPLE

In [49]:
binary_feature_names = [ ]

In [50]:
categorical_feature_names = ['X','Y']

In [51]:
inputs = {}
for name, column in df.items():
  if type(column[0]) == str:
    dtype = tf.string
  elif (name in categorical_feature_names or
        name in binary_feature_names):
    dtype = tf.int64
  else:
    dtype = tf.float32

  inputs[name] = tf.keras.Input(shape=(), name=name, dtype=dtype)

In [52]:
inputs

{'X': <KerasTensor: shape=(None,) dtype=int64 (created by layer 'X')>,
 'Y': <KerasTensor: shape=(None,) dtype=int64 (created by layer 'Y')>,
 'month': <KerasTensor: shape=(None,) dtype=string (created by layer 'month')>,
 'day': <KerasTensor: shape=(None,) dtype=string (created by layer 'day')>,
 'FFMC': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'FFMC')>,
 'DMC': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'DMC')>,
 'DC': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'DC')>,
 'ISI': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'ISI')>,
 'temp': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'temp')>,
 'RH': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'RH')>,
 'wind': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'wind')>,
 'rain': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'rain')>}

For each input you'll apply some transformations using Keras layers and TensorFlow ops. Each feature starts as a batch of scalars (`shape=(batch,)`). The output for each  should be a batch of `tf.float32` vectors (`shape=(batch, n)`). The last step will concatenate all those vectors together.


In [53]:
preprocessed = []

for name in binary_feature_names:
  inp = inputs[name]
  inp = inp[:, tf.newaxis]
  float_value = tf.cast(inp, tf.float32)
  preprocessed.append(float_value)

preprocessed

[]

In [54]:
normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(stack_dict(dict(numeric_features)))

In [55]:
numeric_inputs = {}
for name in numeric_feature_names:
  numeric_inputs[name]=inputs[name]

numeric_inputs = stack_dict(numeric_inputs)
numeric_normalized = normalizer(numeric_inputs)

preprocessed.append(numeric_normalized)

preprocessed

[<KerasTensor: shape=(None, 4) dtype=float32 (created by layer 'normalization_3')>]

In [56]:
vocab = ['a','b','c']
lookup = tf.keras.layers.StringLookup(vocabulary=vocab, output_mode='one_hot')
lookup(['c','a','a','b','zzz'])

<tf.Tensor: shape=(5, 4), dtype=float32, numpy=
array([[0., 0., 0., 1.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [1., 0., 0., 0.]], dtype=float32)>

In [57]:
vocab = [1,4,7,99]
lookup = tf.keras.layers.IntegerLookup(vocabulary=vocab, output_mode='one_hot')

lookup([-1,4,1])

<tf.Tensor: shape=(3, 5), dtype=float32, numpy=
array([[1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0.]], dtype=float32)>

In [58]:
for name in categorical_feature_names:
  vocab = sorted(set(df[name]))
  print(f'name: {name}')
  print(f'vocab: {vocab}\n')

  if type(vocab[0]) is str:
    lookup = tf.keras.layers.StringLookup(vocabulary=vocab, output_mode='one_hot')
  else:
    lookup = tf.keras.layers.IntegerLookup(vocabulary=vocab, output_mode='one_hot')

  x = inputs[name][:, tf.newaxis]
  x = lookup(x)
  preprocessed.append(x)

name: X
vocab: [1, 2, 3, 4, 5, 6, 7, 8, 9]

name: Y
vocab: [2, 3, 4, 5, 6, 8, 9]



In [59]:
preprocessed

[<KerasTensor: shape=(None, 4) dtype=float32 (created by layer 'normalization_3')>,
 <KerasTensor: shape=(None, 10) dtype=float32 (created by layer 'integer_lookup_1')>,
 <KerasTensor: shape=(None, 8) dtype=float32 (created by layer 'integer_lookup_2')>]

In [60]:
preprocessed_result = tf.concat(preprocessed, axis=-1)
preprocessed_result

<KerasTensor: shape=(None, 22) dtype=float32 (created by layer 'tf.concat_1')>

In [61]:
preprocessor = tf.keras.Model(inputs, preprocessed_result)

In [62]:
tf.keras.utils.plot_model(preprocessor, rankdir="LR", show_shapes=True)

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [63]:
preprocessor(dict(df.iloc[:1]))

<tf.Tensor: shape=(1, 22), dtype=float32, numpy=
array([[ 0.41172418, -0.07326832, -1.8426397 ,  1.498614  ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  1.        ,  0.        ,
         0.        ,  0.        ]], dtype=float32)>

In [64]:
body = tf.keras.Sequential([
  tf.keras.layers.Dense(10, activation='relu'),
  tf.keras.layers.Dense(10, activation='relu'),
  tf.keras.layers.Dense(1)
])

In [65]:
inputs

{'X': <KerasTensor: shape=(None,) dtype=int64 (created by layer 'X')>,
 'Y': <KerasTensor: shape=(None,) dtype=int64 (created by layer 'Y')>,
 'month': <KerasTensor: shape=(None,) dtype=string (created by layer 'month')>,
 'day': <KerasTensor: shape=(None,) dtype=string (created by layer 'day')>,
 'FFMC': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'FFMC')>,
 'DMC': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'DMC')>,
 'DC': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'DC')>,
 'ISI': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'ISI')>,
 'temp': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'temp')>,
 'RH': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'RH')>,
 'wind': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'wind')>,
 'rain': <KerasTensor: shape=(None,) dtype=float32 (created by layer 'rain')>}

In [66]:
x = preprocessor(inputs)
x

<KerasTensor: shape=(None, 22) dtype=float32 (created by layer 'model_1')>

In [67]:
result = body(x)
result

<KerasTensor: shape=(None, 1) dtype=float32 (created by layer 'sequential_3')>

In [68]:
model = tf.keras.Model(inputs, result)

model.compile(optimizer='adam',
                loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
                metrics=['accuracy'])

In [69]:
history = model.fit(dict(df), target, epochs=5, batch_size=BATCH_SIZE)

Epoch 1/5
259/259 [==============================] - 1s 1ms/step - loss: -3.1436 - accuracy: 0.3965
Epoch 2/5
259/259 [==============================] - 0s 1ms/step - loss: -17.5179 - accuracy: 0.1335
Epoch 3/5
259/259 [==============================] - 0s 1ms/step - loss: -35.7738 - accuracy: 0.0097
Epoch 4/5
259/259 [==============================] - 0s 1ms/step - loss: -63.8575 - accuracy: 0.0000e+00
Epoch 5/5
259/259 [==============================] - 0s 1ms/step - loss: -125.9613 - accuracy: 0.0000e+00


In [70]:
ds = tf.data.Dataset.from_tensor_slices((
    dict(df),
    target
))

ds = ds.batch(BATCH_SIZE)

In [71]:
import pprint

for x, y in ds.take(1):
  pprint.pprint(x)
  print()
  print(y)

{'DC': <tf.Tensor: shape=(2,), dtype=float64, numpy=array([ 94.3, 669.1])>,
 'DMC': <tf.Tensor: shape=(2,), dtype=float64, numpy=array([26.2, 35.4])>,
 'FFMC': <tf.Tensor: shape=(2,), dtype=float64, numpy=array([86.2, 90.6])>,
 'ISI': <tf.Tensor: shape=(2,), dtype=float64, numpy=array([5.1, 6.7])>,
 'RH': <tf.Tensor: shape=(2,), dtype=int64, numpy=array([51, 33], dtype=int64)>,
 'X': <tf.Tensor: shape=(2,), dtype=int64, numpy=array([7, 7], dtype=int64)>,
 'Y': <tf.Tensor: shape=(2,), dtype=int64, numpy=array([5, 4], dtype=int64)>,
 'day': <tf.Tensor: shape=(2,), dtype=string, numpy=array([b'fri', b'tue'], dtype=object)>,
 'month': <tf.Tensor: shape=(2,), dtype=string, numpy=array([b'mar', b'oct'], dtype=object)>,
 'rain': <tf.Tensor: shape=(2,), dtype=float64, numpy=array([0., 0.])>,
 'temp': <tf.Tensor: shape=(2,), dtype=float64, numpy=array([ 8.2, 18. ])>,
 'wind': <tf.Tensor: shape=(2,), dtype=float64, numpy=array([6.7, 0.9])>}

tf.Tensor([0. 0.], shape=(2,), dtype=float64)


In [72]:
history = model.fit(ds, epochs=5)

Epoch 1/5
259/259 [==============================] - 1s 1ms/step - loss: -218.7104 - accuracy: 0.0000e+00
Epoch 2/5
259/259 [==============================] - 0s 1ms/step - loss: -342.4972 - accuracy: 0.0000e+00
Epoch 3/5
259/259 [==============================] - 0s 1ms/step - loss: -506.5128 - accuracy: 0.0000e+00
Epoch 4/5
259/259 [==============================] - 0s 1ms/step - loss: -714.7309 - accuracy: 0.0000e+00
Epoch 5/5
259/259 [==============================] - 0s 1ms/step - loss: -969.1985 - accuracy: 0.0000e+00
